**04-donor-held-out**  
This notebook runs model training + evaluation pipeline on Scheme B, donor-held-out split. We are performing analysis on the filtered dataset only as it could avoid noise from rare cell types. Batch covariates are still considered.  

Structure:  
1. Logistic regression on the filtered dataset (no batch covariates)  
2. Logistic regression on the filtered dataset with batch covariate (site)  

Then, we explore the effect of using the full dataset.  
At the end, we conduct donor ablation testing.

In [1]:
import os
from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)

In [2]:
REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()   # change if needed
os.chdir(REPO_ROOT)

In [3]:
# -------------------------
# global settings
# -------------------------
CELLTYPE_COL = "cell_type"
DONOR_COL = "patient_id"
BATCH_COL = "Site"
RANDOM_STATE = 42

REPRESENTATIONS = {
    "hvg": "hvg",
    "pca": "X_pca",
    "harmony": "X_harmony",
    "scvi": "X_scVI",
}

FILTERED_DATA_PATH = "data/adata_filtered_celltypes.h5ad"
FULL_DATA_PATH = "data/adata_full_celltypes.h5ad"

# -------------------------
# helper functions
# -------------------------
def get_representation(adata, rep_name):
    """
    Return feature matrix for a given representation.
    - hvg uses adata.X
    - other representations use adata.obsm[rep_name]
    """
    if rep_name == "hvg":
        X = adata.X
    else:
        X = adata.obsm[rep_name]

    if hasattr(X, "toarray"):
        X = X.toarray()
    else:
        X = np.asarray(X)

    return X


def get_batch_matrix(obs, batch_col):
    """
    One-hot encode batch/site covariate.
    """
    batch_df = pd.get_dummies(
        obs[batch_col].astype(str),
        prefix=batch_col,
        drop_first=False,
    )
    return batch_df.to_numpy(), batch_df.columns.tolist()


def restrict_to_train_labels(y_train, y_test, X_test, obs_test):
    """
    Remove test cells whose labels are absent in training donors.
    This avoids impossible multiclass prediction targets.
    """
    train_labels = set(pd.Series(y_train).astype(str).unique())
    keep_mask = pd.Series(y_test).astype(str).isin(train_labels).to_numpy()

    y_test = np.asarray(y_test)[keep_mask]
    X_test = X_test[keep_mask]
    obs_test = obs_test.loc[keep_mask].copy()

    return y_test, X_test, obs_test


def save_confusion_matrix(cm, labels, out_file, title, normalize=False):
    """
    Save confusion matrix plot.
    """
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_plot = np.divide(
            cm.astype(float),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )
    else:
        cm_plot = cm

    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_plot, display_labels=labels)
    disp.plot(
        ax=ax,
        xticks_rotation=90,
        cmap="Blues",
        colorbar=True,
        values_format=None,
    )

    if disp.text_ is not None:
        for text in disp.text_.ravel():
            if text is not None:
                text.set_visible(False)

    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()


def summarize_classification(y_true, y_pred, labels):
    """
    Return macro F1, accuracy, confusion matrix, and per-class table.
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    per_class_f1 = pd.DataFrame({
        "cell_type": labels,
        "f1": [report[label]["f1-score"] for label in labels],
        "precision": [report[label]["precision"] for label in labels],
        "recall": [report[label]["recall"] for label in labels],
        "support": [report[label]["support"] for label in labels],
    })

    return {
        "macro_f1": macro_f1,
        "accuracy": acc,
        "cm": cm,
        "per_class_f1": per_class_f1,
    }

In [4]:
# core functions to run logistic regression with donor split and optional batch covariates
def run_donor_split_logreg(
    adata,
    rep_name,
    train_donors,
    test_donors,
    celltype_col=CELLTYPE_COL,
    donor_col=DONOR_COL,
    batch_col=None,
    random_state=RANDOM_STATE,
):
    """
    Train on one donor set and test on a disjoint donor set.
    Optionally append one-hot batch/site covariates.
    """
    obs = adata.obs.copy()
    X_rep = get_representation(adata, rep_name)

    train_mask = obs[donor_col].astype(str).isin(pd.Series(train_donors).astype(str))
    test_mask = obs[donor_col].astype(str).isin(pd.Series(test_donors).astype(str))

    X_train_rep = X_rep[train_mask.to_numpy()]
    X_test_rep = X_rep[test_mask.to_numpy()]

    y_train = obs.loc[train_mask, celltype_col].astype(str).to_numpy()
    y_test = obs.loc[test_mask, celltype_col].astype(str).to_numpy()

    obs_train = obs.loc[train_mask].copy()
    obs_test = obs.loc[test_mask].copy()

    # remove labels that are too rare in training
    train_counts = pd.Series(y_train).value_counts()
    keep_train_labels = train_counts[train_counts >= 2].index

    keep_train_mask = pd.Series(y_train).isin(keep_train_labels).to_numpy()
    X_train_rep = X_train_rep[keep_train_mask]
    y_train = y_train[keep_train_mask]
    obs_train = obs_train.loc[keep_train_mask].copy()

    # keep only test labels seen in training
    y_test, X_test_rep, obs_test = restrict_to_train_labels(
        y_train=y_train,
        y_test=y_test,
        X_test=X_test_rep,
        obs_test=obs_test,
    )

    if len(y_test) == 0:
        raise ValueError("No valid test cells remain after restricting to training labels.")

    # optional batch features
    if batch_col is not None:
        train_batch_df = pd.get_dummies(
            obs_train[batch_col].astype(str),
            prefix=batch_col,
            drop_first=False,
        )
        test_batch_df = pd.get_dummies(
            obs_test[batch_col].astype(str),
            prefix=batch_col,
            drop_first=False,
        )

        # align columns so train/test match exactly
        train_batch_df, test_batch_df = train_batch_df.align(
            test_batch_df,
            join="outer",
            axis=1,
            fill_value=0,
        )

        X_train_batch = train_batch_df.to_numpy()
        X_test_batch = test_batch_df.to_numpy()
        batch_feature_names = train_batch_df.columns.tolist()
    else:
        X_train_batch = None
        X_test_batch = None
        batch_feature_names = []

    # scale only representation features
    scaler = StandardScaler()
    X_train_rep = scaler.fit_transform(X_train_rep)
    X_test_rep = scaler.transform(X_test_rep)

    if X_train_batch is not None:
        X_train = np.hstack([X_train_rep, X_train_batch])
        X_test = np.hstack([X_test_rep, X_test_batch])
    else:
        X_train = X_train_rep
        X_test = X_test_rep

    clf = LogisticRegression(
        max_iter=5000,
        random_state=random_state,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    labels = np.unique(np.concatenate([y_train, y_test]))

    summary = summarize_classification(y_test, y_pred, labels)

    return {
        "model": clf,
        "scaler": scaler,
        "train_donors": list(train_donors),
        "test_donors": list(test_donors),
        "y_test": y_test,
        "y_pred": y_pred,
        "labels": labels,
        "macro_f1": summary["macro_f1"],
        "accuracy": summary["accuracy"],
        "cm": summary["cm"],
        "per_class_f1": summary["per_class_f1"],
        "n_train_cells": len(y_train),
        "n_test_cells": len(y_test),
        "n_classes_used": len(labels),
        "batch_feature_names": batch_feature_names,
    }

def make_donor_folds(adata, donor_col=DONOR_COL, n_folds=5, random_state=RANDOM_STATE):
    """
    Split unique donors into folds.
    Returns a list of donor arrays.
    """
    donors = np.array(sorted(adata.obs[donor_col].astype(str).unique()))
    rng = np.random.default_rng(random_state)
    shuffled = donors.copy()
    rng.shuffle(shuffled)

    donor_folds = np.array_split(shuffled, n_folds)
    return donor_folds

def run_donor_cv_experiment(
    adata,
    rep_name,
    results_dir,
    scheme_label,
    donor_col=DONOR_COL,
    batch_col=None,
    n_folds=5,
    random_state=RANDOM_STATE,
):
    """
    Run donor-held-out cross-validation for one representation.
    Saves fold-level outputs and returns summary metrics.
    """
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    donor_folds = make_donor_folds(
        adata,
        donor_col=donor_col,
        n_folds=n_folds,
        random_state=random_state,
    )

    fold_metrics = []
    all_pred_rows = []
    per_class_tables = []

    cm_sum = None
    final_labels = None

    for fold_idx in range(len(donor_folds)):
        test_donors = donor_folds[fold_idx]
        train_donors = np.concatenate(
            [donor_folds[j] for j in range(len(donor_folds)) if j != fold_idx]
        )

        print(
            f"  Fold {fold_idx + 1}/{len(donor_folds)} | "
            f"train donors: {len(train_donors)} | test donors: {len(test_donors)}"
        )

        res = run_donor_split_logreg(
            adata=adata,
            rep_name=rep_name,
            train_donors=train_donors,
            test_donors=test_donors,
            donor_col=donor_col,
            batch_col=batch_col,
            random_state=random_state,
        )

        print(f"    Macro F1: {res['macro_f1']:.4f}")
        print(f"    Accuracy: {res['accuracy']:.4f}")

        # save fold confusion matrix csv
        cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
        cm_df.to_csv(results_dir / f"{scheme_label}_{rep_name}_fold{fold_idx+1}_confusion_matrix.csv")

        # save fold normalized confusion matrix plot
        title = f"{scheme_label} - {rep_name} - Fold {fold_idx+1} (row-normalized)"
        save_confusion_matrix(
            res["cm"],
            res["labels"],
            results_dir / f"{scheme_label}_{rep_name}_fold{fold_idx+1}_confusion_matrix_normalized.png",
            title=title,
            normalize=True,
        )

        # save fold predictions
        pred_df = pd.DataFrame({
            "fold": fold_idx + 1,
            "y_true": res["y_test"],
            "y_pred": res["y_pred"],
        })
        pred_df.to_csv(
            results_dir / f"{scheme_label}_{rep_name}_fold{fold_idx+1}_predictions.csv",
            index=False,
        )
        all_pred_rows.append(pred_df)

        # save fold per-class metrics
        fold_per_class = res["per_class_f1"].copy()
        fold_per_class["fold"] = fold_idx + 1
        fold_per_class.to_csv(
            results_dir / f"{scheme_label}_{rep_name}_fold{fold_idx+1}_per_class_f1.csv",
            index=False,
        )
        per_class_tables.append(fold_per_class)

        # fold summary row
        fold_metrics.append({
            "fold": fold_idx + 1,
            "representation": rep_name,
            "macro_f1": res["macro_f1"],
            "accuracy": res["accuracy"],
            "n_train_cells": res["n_train_cells"],
            "n_test_cells": res["n_test_cells"],
            "n_classes_used": res["n_classes_used"],
            "n_train_donors": len(res["train_donors"]),
            "n_test_donors": len(res["test_donors"]),
            "batch_covariate": batch_col if batch_col is not None else "None",
            "n_batch_features": len(res["batch_feature_names"]),
        })

        # accumulate confusion matrix across folds
        if cm_sum is None:
            cm_sum = res["cm"].copy()
            final_labels = res["labels"]
        else:
            cm_sum = cm_sum + res["cm"]

    # save combined outputs
    fold_metrics_df = pd.DataFrame(fold_metrics)
    fold_metrics_df.to_csv(results_dir / f"{scheme_label}_{rep_name}_fold_metrics.csv", index=False)

    all_preds_df = pd.concat(all_pred_rows, ignore_index=True)
    all_preds_df.to_csv(results_dir / f"{scheme_label}_{rep_name}_all_predictions.csv", index=False)

    per_class_all_df = pd.concat(per_class_tables, ignore_index=True)
    per_class_all_df.to_csv(results_dir / f"{scheme_label}_{rep_name}_all_per_class_f1.csv", index=False)

    # mean per-class performance across folds
    per_class_mean_df = (
        per_class_all_df
        .groupby("cell_type", as_index=False)[["f1", "precision", "recall", "support"]]
        .mean()
        .sort_values("f1", ascending=False)
        .reset_index(drop=True)
    )
    per_class_mean_df.to_csv(
        results_dir / f"{scheme_label}_{rep_name}_mean_per_class_f1.csv",
        index=False,
    )

    # save aggregated confusion matrix
    cm_sum_df = pd.DataFrame(cm_sum, index=final_labels, columns=final_labels)
    cm_sum_df.to_csv(results_dir / f"{scheme_label}_{rep_name}_confusion_matrix_sum.csv")

    save_confusion_matrix(
        cm_sum,
        final_labels,
        results_dir / f"{scheme_label}_{rep_name}_confusion_matrix_sum_normalized.png",
        title=f"{scheme_label} - {rep_name} (summed across folds, row-normalized)",
        normalize=True,
    )

    # final summary for this representation
    summary_row = {
        "representation": rep_name,
        "macro_f1_mean": fold_metrics_df["macro_f1"].mean(),
        "macro_f1_std": fold_metrics_df["macro_f1"].std(ddof=1),
        "accuracy_mean": fold_metrics_df["accuracy"].mean(),
        "accuracy_std": fold_metrics_df["accuracy"].std(ddof=1),
        "n_folds": len(fold_metrics_df),
        "mean_train_cells": fold_metrics_df["n_train_cells"].mean(),
        "mean_test_cells": fold_metrics_df["n_test_cells"].mean(),
        "mean_n_classes_used": fold_metrics_df["n_classes_used"].mean(),
        "batch_covariate": batch_col if batch_col is not None else "None",
        "mean_n_batch_features": fold_metrics_df["n_batch_features"].mean(),
    }

    return {
        "summary_row": summary_row,
        "fold_metrics_df": fold_metrics_df,
        "per_class_mean_df": per_class_mean_df,
        "cm_sum": cm_sum,
        "labels": final_labels,
    }

**Case 1: Filtered dataset, Logistic regression without a batch covariate**

In [6]:
adata = sc.read_h5ad(FILTERED_DATA_PATH)
results_dir = Path("results/schemeB_case1")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme B Case 1 for {rep_label} ({rep_key})...")

    out = run_donor_cv_experiment(
        adata=adata,
        rep_name=rep_key,
        results_dir=results_dir,
        scheme_label="schemeB_case1",
        batch_col=None,
        n_folds=5,
        random_state=RANDOM_STATE,
    )

    metrics_rows.append({
        "scheme": "donor_held_out",
        "dataset": "filtered",
        "batch": "no",
        **out["summary_row"],
    })

metrics_case1 = pd.DataFrame(metrics_rows)
metrics_case1 = metrics_case1.sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)
metrics_case1.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme B Case 1 metrics:")
print(metrics_case1)

Running Scheme B Case 1 for hvg (hvg)...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7600
    Accuracy: 0.8227
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7390
    Accuracy: 0.7866
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7906
    Accuracy: 0.8231
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7600
    Accuracy: 0.7699
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7765
    Accuracy: 0.8102
Running Scheme B Case 1 for pca (X_pca)...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7916
    Accuracy: 0.8505
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7278
    Accuracy: 0.7662
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.8098
    Accuracy: 0.8437
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7638
    Accuracy: 0.7837
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7851
    Accuracy: 0.8257
Running Scheme B Case 1 fo

**Case 2: Filtered dataset, Logistic regression with a batch covariate (site)**

In [7]:
adata = sc.read_h5ad(FILTERED_DATA_PATH)
results_dir = Path("results/schemeB_case2")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme B Case 2 for {rep_label} ({rep_key}) with batch covariate {BATCH_COL}...")

    out = run_donor_cv_experiment(
        adata=adata,
        rep_name=rep_key,
        results_dir=results_dir,
        scheme_label="schemeB_case2",
        batch_col=BATCH_COL,
        n_folds=5,
        random_state=RANDOM_STATE,
    )

    metrics_rows.append({
        "scheme": "donor_held_out_batchcov",
        "dataset": "filtered",
        "batch": "yes",
        **out["summary_row"],
    })

metrics_case2 = pd.DataFrame(metrics_rows)
metrics_case2 = metrics_case2.sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)
metrics_case2.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme B Case 2 metrics:")
print(metrics_case2)

Running Scheme B Case 2 for hvg (hvg) with batch covariate Site...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7590
    Accuracy: 0.8222
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7364
    Accuracy: 0.7824
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7898
    Accuracy: 0.8227
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7590
    Accuracy: 0.7693
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7775
    Accuracy: 0.8113
Running Scheme B Case 2 for pca (X_pca) with batch covariate Site...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7907
    Accuracy: 0.8477
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.7263
    Accuracy: 0.7643
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.8162
    Accuracy: 0.8487
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.7663
    Accuracy: 0.7859
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.

In [9]:
# summary across 2 main cases
summary_df = pd.concat([metrics_case1, metrics_case2], ignore_index=True)
summary_df.to_csv("results/tables/schemeB_summary.csv", index=False)

print("\nCombined Scheme B summary:")
summary_df


Combined Scheme B summary:


,scheme,dataset,batch,representation,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,n_folds,mean_train_cells,mean_test_cells,mean_n_classes_used,batch_covariate,mean_n_batch_features
0,donor_held_out,filtered,no,X_pca,0.775636,0.031406,0.813954,0.037290,5,8085.6,2021.4,13.0,None,0.0
1,donor_held_out,filtered,no,hvg,0.765229,0.019433,0.802494,0.023497,5,8085.6,2021.4,13.0,None,0.0
2,donor_held_out,filtered,no,X_harmony,0.748225,0.030225,0.788531,0.035855,5,8085.6,2021.4,13.0,None,0.0
3,donor_held_out,filtered,no,X_scVI,0.744218,0.034787,0.787228,0.040640,5,8085.6,2021.4,13.0,None,0.0
4,donor_held_out_batchcov,filtered,yes,X_pca,0.776870,0.033406,0.814029,0.037734,5,8085.6,2021.4,13.0,Site,2.0
5,donor_held_out_batchcov,filtered,yes,hvg,0.764340,0.020338,0.801585,0.024353,5,8085.6,2021.4,13.0,Site,2.0
6,donor_held_out_batchcov,filtered,yes,X_harmony,0.747530,0.042606,0.791621,0.041989,5,8085.6,2021.4,13.0,Site,2.0
7,donor_held_out_batchcov,filtered,yes,X_scVI,0.743975,0.038099,0.790279,0.038479,5,8085.6,2021.4,13.0,Site,2.0


**Full Dataset Comparison**: Does including rare cell types hurt performance?

In [11]:
adata = sc.read_h5ad(FULL_DATA_PATH)

results_dir = Path("results/schemeB_full_comparison")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running full dataset comparison for {rep_label} ({rep_key})...")

    out = run_donor_cv_experiment(
        adata=adata,
        rep_name=rep_key,
        results_dir=results_dir,
        scheme_label="schemeB_full",
        batch_col=None,        
        n_folds=5,
        random_state=RANDOM_STATE,
    )

    metrics_rows.append({
        "scheme": "donor_held_out",
        "dataset": "full",
        "batch": "no",
        **out["summary_row"],
    })

metrics_full = pd.DataFrame(metrics_rows)
metrics_full = metrics_full.sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)
metrics_full.to_csv(results_dir / "metrics.csv", index=False)

print("\nFull dataset comparison metrics:")
metrics_full

Running full dataset comparison for hvg (hvg)...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.5172
    Accuracy: 0.7832
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.5145
    Accuracy: 0.7316
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.5529
    Accuracy: 0.7792
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.5406
    Accuracy: 0.7421
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.5398
    Accuracy: 0.7680
Running full dataset comparison for pca (X_pca)...
  Fold 1/5 | train donors: 18 | test donors: 5
    Macro F1: 0.4673
    Accuracy: 0.8115
  Fold 2/5 | train donors: 18 | test donors: 5
    Macro F1: 0.4432
    Accuracy: 0.7227
  Fold 3/5 | train donors: 18 | test donors: 5
    Macro F1: 0.5009
    Accuracy: 0.7955
  Fold 4/5 | train donors: 19 | test donors: 4
    Macro F1: 0.5155
    Accuracy: 0.7492
  Fold 5/5 | train donors: 19 | test donors: 4
    Macro F1: 0.5299
    Accuracy: 0.7827
Running fu

,scheme,dataset,batch,representation,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,n_folds,mean_train_cells,mean_test_cells,mean_n_classes_used,batch_covariate,mean_n_batch_features
0,donor_held_out,full,no,hvg,0.533016,0.016528,0.760840,0.022875,5,9031.2,2257.8,40.0,None,0.0
1,donor_held_out,full,no,X_scVI,0.521743,0.030708,0.753611,0.035246,5,9031.2,2257.8,40.0,None,0.0
2,donor_held_out,full,no,X_pca,0.491358,0.035545,0.772323,0.035959,5,9031.2,2257.8,40.0,None,0.0
3,donor_held_out,full,no,X_harmony,0.465628,0.033495,0.749319,0.033094,5,9031.2,2257.8,40.0,None,0.0


In [12]:
# combine results
comparison_df = pd.concat(
    [metrics_case1, metrics_full],
    ignore_index=True
)

comparison_df.to_csv("results/tables/schemeB_filtered_vs_full.csv", index=False)

print("\nFiltered vs Full comparison:")
comparison_df


Filtered vs Full comparison:


,scheme,dataset,batch,representation,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,n_folds,mean_train_cells,mean_test_cells,mean_n_classes_used,batch_covariate,mean_n_batch_features
0,donor_held_out,filtered,no,X_pca,0.775636,0.031406,0.813954,0.037290,5,8085.6,2021.4,13.0,None,0.0
1,donor_held_out,filtered,no,hvg,0.765229,0.019433,0.802494,0.023497,5,8085.6,2021.4,13.0,None,0.0
2,donor_held_out,filtered,no,X_harmony,0.748225,0.030225,0.788531,0.035855,5,8085.6,2021.4,13.0,None,0.0
3,donor_held_out,filtered,no,X_scVI,0.744218,0.034787,0.787228,0.040640,5,8085.6,2021.4,13.0,None,0.0
4,donor_held_out,full,no,hvg,0.533016,0.016528,0.760840,0.022875,5,9031.2,2257.8,40.0,None,0.0
5,donor_held_out,full,no,X_scVI,0.521743,0.030708,0.753611,0.035246,5,9031.2,2257.8,40.0,None,0.0
6,donor_held_out,full,no,X_pca,0.491358,0.035545,0.772323,0.035959,5,9031.2,2257.8,40.0,None,0.0
7,donor_held_out,full,no,X_harmony,0.465628,0.033495,0.749319,0.033094,5,9031.2,2257.8,40.0,None,0.0


**Donor Ablation Analysis**

In [5]:
# helper function
def sample_train_donors(all_donors, k, random_state=42):
    rng = np.random.default_rng(random_state)
    return rng.choice(all_donors, size=k, replace=False)

adata = sc.read_h5ad(FILTERED_DATA_PATH) # read in FILTERED DATA for donor ablation experiment
all_donors = np.array(sorted(adata.obs[DONOR_COL].astype(str).unique()))
k_values = [3, 5, 8, 10, 15, 20] # 23 donors in total, so leave at least 3 for testing
n_repeats = 5

results = []

# iterate, run for all representations
for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"\nRunning donor ablation for {rep_label} ({rep_key})")

    for k in k_values:
        for repeat in range(n_repeats):
            seed = RANDOM_STATE + 100 * k + repeat

            train_donors = sample_train_donors(
                all_donors,
                k,
                random_state=seed,
            )

            test_donors = np.array([d for d in all_donors if d not in train_donors])

            res = run_donor_split_logreg(
                adata=adata,
                rep_name=rep_key,
                train_donors=train_donors,
                test_donors=test_donors,
                donor_col=DONOR_COL,
                batch_col=None,   #  no batch covariate for ablation
                random_state=RANDOM_STATE,
            )

            results.append({
                "representation": rep_label,
                "rep_key": rep_key,
                "k_train_donors": k,
                "repeat": repeat + 1,
                "macro_f1": res["macro_f1"],
                "accuracy": res["accuracy"],
                "n_train_cells": res["n_train_cells"],
                "n_test_cells": res["n_test_cells"],
                "n_classes_used": res["n_classes_used"],
            })

            print(
                f"  k={k:2d}, repeat={repeat+1}: "
                f"macro_f1={res['macro_f1']:.4f}, acc={res['accuracy']:.4f}"
            )

ablation_df = pd.DataFrame(results)
ablation_df.to_csv("results/tables/donor_ablation_raw.csv", index=False)

ablation_summary = (
    ablation_df
    .groupby(["representation", "k_train_donors"], as_index=False)
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        mean_train_cells=("n_train_cells", "mean"),
        mean_test_cells=("n_test_cells", "mean"),
        mean_n_classes_used=("n_classes_used", "mean"),
    )
)

ablation_summary.to_csv("results/tables/donor_ablation_summary.csv", index=False)

ablation_summary


Running donor ablation for hvg (hvg)
  k= 3, repeat=1: macro_f1=0.6132, acc=0.6359
  k= 3, repeat=2: macro_f1=0.7122, acc=0.7395
  k= 3, repeat=3: macro_f1=0.5518, acc=0.6243
  k= 3, repeat=4: macro_f1=0.6900, acc=0.7501
  k= 3, repeat=5: macro_f1=0.7013, acc=0.7327
  k= 5, repeat=1: macro_f1=0.7234, acc=0.7746
  k= 5, repeat=2: macro_f1=0.6730, acc=0.7174
  k= 5, repeat=3: macro_f1=0.7132, acc=0.7682
  k= 5, repeat=4: macro_f1=0.7175, acc=0.7580
  k= 5, repeat=5: macro_f1=0.7335, acc=0.7664
  k= 8, repeat=1: macro_f1=0.7497, acc=0.8012
  k= 8, repeat=2: macro_f1=0.7341, acc=0.7467
  k= 8, repeat=3: macro_f1=0.7577, acc=0.7927
  k= 8, repeat=4: macro_f1=0.7430, acc=0.7844
  k= 8, repeat=5: macro_f1=0.7625, acc=0.8024
  k=10, repeat=1: macro_f1=0.7682, acc=0.8048
  k=10, repeat=2: macro_f1=0.7515, acc=0.8202
  k=10, repeat=3: macro_f1=0.7444, acc=0.7848
  k=10, repeat=4: macro_f1=0.7554, acc=0.8048
  k=10, repeat=5: macro_f1=0.7686, acc=0.8059
  k=15, repeat=1: macro_f1=0.7569, acc=0.7

,representation,k_train_donors,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,mean_train_cells,mean_test_cells,mean_n_classes_used
0,harmony,3,0.672019,0.045301,0.712732,0.043351,1297.2,8809.8,13.0
1,harmony,5,0.714775,0.015743,0.758139,0.018409,2161.0,7946.0,13.0
2,harmony,8,0.736220,0.009037,0.772962,0.018187,3509.6,6597.4,13.0
3,harmony,10,0.743356,0.008474,0.791493,0.009600,4401.6,5705.4,13.0
4,harmony,15,0.757405,0.016730,0.796937,0.016833,6581.6,3525.4,13.0
5,harmony,20,0.726524,0.015108,0.770169,0.039392,8777.8,1329.2,13.0
6,hvg,3,0.653710,0.068984,0.696490,0.061090,1297.2,8809.8,13.0
7,hvg,5,0.712121,0.023163,0.756918,0.022867,2161.0,7946.0,13.0
8,hvg,8,0.749423,0.011331,0.785494,0.022850,3509.6,6597.4,13.0
9,hvg,10,0.757629,0.010610,0.804111,0.012599,4401.6,5705.4,13.0


**Additional Exploration: Train on all donors of one site, and test on all donors of other site**

In [5]:
# =========================================================
# Cross-site generalization analysis
# Train on one site, test on the other, then flip direction
# Uses filtered dataset only
# =========================================================

adata = sc.read_h5ad(FILTERED_DATA_PATH)

results_dir = Path("results/site_generalization")
results_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# inspect site / donor mapping
# -------------------------
site_donor_df = (
    adata.obs[[DONOR_COL, BATCH_COL]]
    .drop_duplicates()
    .sort_values([BATCH_COL, DONOR_COL])
    .reset_index(drop=True)
)

print("Unique donor-site pairs:")
display(site_donor_df)

site_to_donors = (
    site_donor_df
    .groupby(BATCH_COL)[DONOR_COL]
    .apply(lambda x: np.array(sorted(x.astype(str).unique())))
    .to_dict()
)

site_names = sorted(site_to_donors.keys())
if len(site_names) != 2:
    raise ValueError(
        f"Expected exactly 2 sites for cross-site analysis, found {len(site_names)}: {site_names}"
    )

site_A, site_B = site_names
donors_A = site_to_donors[site_A]
donors_B = site_to_donors[site_B]

print(f"\nSite A: {site_A} | n_donors = {len(donors_A)}")
print(donors_A)

print(f"\nSite B: {site_B} | n_donors = {len(donors_B)}")
print(donors_B)

# -------------------------
# helper to run one direction
# -------------------------
def run_cross_site_experiment(
    adata,
    rep_name,
    train_site_name,
    test_site_name,
    train_donors,
    test_donors,
    results_dir,
    scheme_label="cross_site",
):
    """
    Train on donors from one site and test on donors from the other site.
    Saves confusion matrix + per-class metrics for each direction.
    """
    out = run_donor_split_logreg(
        adata=adata,
        rep_name=rep_name,
        train_donors=train_donors,
        test_donors=test_donors,
        donor_col=DONOR_COL,
        batch_col=None,   # do not include site covariate here
        random_state=RANDOM_STATE,
    )

    direction_label = f"{train_site_name}_to_{test_site_name}".replace(" ", "_")
    prefix = f"{scheme_label}_{rep_name}_{direction_label}"

    # save confusion matrix csv
    cm_df = pd.DataFrame(out["cm"], index=out["labels"], columns=out["labels"])
    cm_df.to_csv(results_dir / f"{prefix}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        out["cm"],
        out["labels"],
        results_dir / f"{prefix}_confusion_matrix_normalized.png",
        title=f"{rep_name}: {train_site_name} → {test_site_name} (row-normalized)",
        normalize=True,
    )

    # save per-class metrics
    per_class_df = out["per_class_f1"].copy()
    per_class_df["representation"] = rep_name
    per_class_df["train_site"] = train_site_name
    per_class_df["test_site"] = test_site_name
    per_class_df.to_csv(results_dir / f"{prefix}_per_class_f1.csv", index=False)

    # save predictions
    pred_df = pd.DataFrame({
        "representation": rep_name,
        "train_site": train_site_name,
        "test_site": test_site_name,
        "y_true": out["y_test"],
        "y_pred": out["y_pred"],
    })
    pred_df.to_csv(results_dir / f"{prefix}_predictions.csv", index=False)

    return {
        "representation": rep_name,
        "train_site": train_site_name,
        "test_site": test_site_name,
        "direction": f"{train_site_name} → {test_site_name}",
        "macro_f1": out["macro_f1"],
        "accuracy": out["accuracy"],
        "n_train_donors": len(train_donors),
        "n_test_donors": len(test_donors),
        "n_train_cells": out["n_train_cells"],
        "n_test_cells": out["n_test_cells"],
        "n_classes_used": out["n_classes_used"],
    }

# -------------------------
# run both directions for all representations
# -------------------------
metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"\nRunning cross-site generalization for {rep_label} ({rep_key})")

    # site A -> site B
    res_AB = run_cross_site_experiment(
        adata=adata,
        rep_name=rep_key,
        train_site_name=site_A,
        test_site_name=site_B,
        train_donors=donors_A,
        test_donors=donors_B,
        results_dir=results_dir,
        scheme_label="cross_site",
    )
    metrics_rows.append(res_AB)
    print(
        f"  {site_A} -> {site_B}: "
        f"macro_f1={res_AB['macro_f1']:.4f}, acc={res_AB['accuracy']:.4f}"
    )

    # site B -> site A
    res_BA = run_cross_site_experiment(
        adata=adata,
        rep_name=rep_key,
        train_site_name=site_B,
        test_site_name=site_A,
        train_donors=donors_B,
        test_donors=donors_A,
        results_dir=results_dir,
        scheme_label="cross_site",
    )
    metrics_rows.append(res_BA)
    print(
        f"  {site_B} -> {site_A}: "
        f"macro_f1={res_BA['macro_f1']:.4f}, acc={res_BA['accuracy']:.4f}"
    )

# -------------------------
# save summary tables
# -------------------------
cross_site_df = pd.DataFrame(metrics_rows)

# clean representation names for plotting
cross_site_df["representation_clean"] = cross_site_df["representation"].replace({
    "hvg": "hvg",
    "X_pca": "pca",
    "X_harmony": "harmony",
    "X_scVI": "scvi",
})

cross_site_df = cross_site_df.sort_values(
    ["representation_clean", "train_site", "test_site"]
).reset_index(drop=True)

cross_site_df.to_csv(results_dir / "cross_site_metrics.csv", index=False)
cross_site_df.to_csv("results/tables/cross_site_summary.csv", index=False)

print("\nCross-site summary:")
display(cross_site_df)

Unique donor-site pairs:


,patient_id,Site
0,CV0902,Cambridge
1,CV0904,Cambridge
2,CV0911,Cambridge
3,CV0915,Cambridge
4,CV0917,Cambridge
5,CV0926,Cambridge
6,CV0929,Cambridge
7,CV0934,Cambridge
8,CV0939,Cambridge
9,CV0940,Cambridge



Site A: Cambridge | n_donors = 11
['CV0902' 'CV0904' 'CV0911' 'CV0915' 'CV0917' 'CV0926' 'CV0929' 'CV0934'
 'CV0939' 'CV0940' 'CV0944']

Site B: Ncl | n_donors = 12
['MH8919176' 'MH8919177' 'MH8919178' 'MH8919179' 'MH8919226' 'MH8919227'
 'MH8919282' 'MH8919283' 'MH8919332' 'MH8919333' 'newcastle65'
 'newcastle74']

Running cross-site generalization for hvg (hvg)


/tmp/ipykernel_1722873/3606501132.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  site_donor_df


  Cambridge -> Ncl: macro_f1=0.7147, acc=0.7941
  Ncl -> Cambridge: macro_f1=0.6529, acc=0.6491

Running cross-site generalization for pca (X_pca)
  Cambridge -> Ncl: macro_f1=0.7158, acc=0.8038
  Ncl -> Cambridge: macro_f1=0.6755, acc=0.6739

Running cross-site generalization for harmony (X_harmony)
  Cambridge -> Ncl: macro_f1=0.7094, acc=0.7811
  Ncl -> Cambridge: macro_f1=0.6629, acc=0.6592

Running cross-site generalization for scvi (X_scVI)
  Cambridge -> Ncl: macro_f1=0.6963, acc=0.7705
  Ncl -> Cambridge: macro_f1=0.6546, acc=0.6474

Cross-site summary:


,representation,train_site,test_site,direction,macro_f1,accuracy,n_train_donors,n_test_donors,n_train_cells,n_test_cells,n_classes_used,representation_clean
0,X_harmony,Cambridge,Ncl,Cambridge → Ncl,0.709438,0.781061,11,12,4827,5280,13,harmony
1,X_harmony,Ncl,Cambridge,Ncl → Cambridge,0.662947,0.659209,12,11,5280,4827,13,harmony
2,hvg,Cambridge,Ncl,Cambridge → Ncl,0.714689,0.794129,11,12,4827,5280,13,hvg
3,hvg,Ncl,Cambridge,Ncl → Cambridge,0.652857,0.649057,12,11,5280,4827,13,hvg
4,X_pca,Cambridge,Ncl,Cambridge → Ncl,0.715772,0.803788,11,12,4827,5280,13,pca
5,X_pca,Ncl,Cambridge,Ncl → Cambridge,0.675472,0.673918,12,11,5280,4827,13,pca
6,X_scVI,Cambridge,Ncl,Cambridge → Ncl,0.696264,0.770455,11,12,4827,5280,13,scvi
7,X_scVI,Ncl,Cambridge,Ncl → Cambridge,0.654639,0.647400,12,11,5280,4827,13,scvi
